In [2]:
!pip install -q -U transformers datasets bitsandbytes  trl peft  huggingface_hub

In [ ]:
from huggingface_hub import login

# Replace with your actual Hugging Face API token
login(token="hf_**********************")


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch


name_name = "Qwen/Qwen2.5-1.5B-Instruct"

config_8bit = BitsAndBytesConfig(load_in_8bit=True)


model_8bit = AutoModelForCausalLM.from_pretrained(

                                                  name_name,
                                                  quantization_config=config_8bit,
                                                  trust_remote_code=True,
                                                  device_map="auto",
                                                  )

tokenizer = AutoTokenizer.from_pretrained(name_name, trust_remote_code=True)

2025-09-03 01:01:24.338716: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756861284.362224     329 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756861284.369250     329 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset",split="train[:5000]")
dataset

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 5000
})

In [6]:
dataset[0]

{'prompt': 'Oh, I just saw the best meme - have you seen it?',
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [7]:
example_text = [{'role':'assistant','content':dataset[0]['chosen']}]
example_text

[{'role': 'assistant',
  'content': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣"}]

In [8]:
tokenizer.apply_chat_template(example_text,tokenize=False)

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n"

In [9]:
def apply_chat_temp(example):

    example['prompt'] =[{'role':'user','content':example['prompt']}]

    return example


new_dataset = dataset.map(apply_chat_temp)
new_dataset[0]

{'prompt': [{'content': 'Oh, I just saw the best meme - have you seen it?',
   'role': 'user'}],
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [10]:
new_templete = """<|im_start|>assistant\n{model_answer}<|im_end|>\n"""


def format_dataset(example):

    example['prompt'] = tokenizer.apply_chat_template(example['prompt'],tokenize=False)
    example['chosen'] = new_templete.format(model_answer =example['chosen'])
    example['rejected'] = new_templete.format(model_answer =example['rejected'])

    return example


formatted_dataset = new_dataset.map(format_dataset)
formatted_dataset[0]

{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOh, I just saw the best meme - have you seen it?<|im_end|>\n',
 'chosen': "<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n",
 'rejected': "<|im_start|>assistant\nI'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?<|im_end|>\n"}

In [11]:
model_8bit

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNor

In [12]:
from peft import LoraConfig, get_peft_model

lora_config= LoraConfig(

                        r = 32,
                        lora_alpha = 32,
                        target_modules =["q_proj","k_proj","v_proj","o_proj",
                  "gate_proj","up_proj","down_proj"],
                        lora_dropout = 0.05,
                        bias = "none",
                        task_type = "CAUSAL_LM"
)
lora_model = get_peft_model(model_8bit,lora_config)
lora_model.print_trainable_parameters()

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [13]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [14]:
from trl import DPOTrainer, DPOConfig
dpo_config = DPOConfig(
    output_dir="./results",
    per_device_train_batch_size=2,       # each GPU handles 4
    gradient_accumulation_steps=16,       # effective batch size = 4*2*8 = 64
    num_train_epochs=1,                  # better than limiting with max_steps
    learning_rate=5e-5, # typical for DPO/LoRA
    fp16=True,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=50,
    save_steps=50,                      # save less frequently to save disk
    eval_steps=50,                      # optional, if you set eval_dataset
    beta=0.05,                           # balanced KL penalty (tweak if too rigid/free)
    report_to="none"                     # no wandb/hf logging
)


In [15]:
trainer = DPOTrainer(

                     model = lora_model,
                    # If no `ref_model` is provided, the trainer creates a copy of the current model (without LoRA adapters).,
                     train_dataset = formatted_dataset,
                     processing_class = tokenizer,
                     args = dpo_config
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
50,0.036000
100,0.000000
150,0.000200


TrainOutput(global_step=157, training_loss=0.011541059725824447, metrics={'train_runtime': 4782.8877, 'train_samples_per_second': 1.045, 'train_steps_per_second': 0.033, 'total_flos': 0.0, 'train_loss': 0.011541059725824447, 'epoch': 1.0})

In [16]:
lora_path = "./lora_adapters"
lora_model.save_pretrained(lora_path)

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
lora_path = "./lora_adapters"


name_name = "Qwen/Qwen2.5-1.5B-Instruct"




base_model = AutoModelForCausalLM.from_pretrained(

                                                  name_name,

                                                  trust_remote_code=True,
                                                  device_map="auto",
                                                  )

tokenizer = AutoTokenizer.from_pretrained(name_name, trust_remote_code=True)

In [18]:
from peft import PeftModel

final_model = PeftModel.from_pretrained(base_model,lora_path)
final_model 

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [19]:
final_model = final_model.merge_and_unload()
final_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

In [ ]:
from huggingface_hub import login

# Log in to Hugging Face
login("hf_*******************")  # replace with your actual token

# Define your repository name
repo_name = "Harsha901/qwen2.5-1.5B-dpo-finetuned"

# Push the model
final_model.push_to_hub(repo_name)

# Push the tokenizer to the same repo
tokenizer.push_to_hub(repo_name)


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...4z/model-00001-of-00002.safetensors:   1%|          | 33.5MB / 5.00GB            

  ...4z/model-00002-of-00002.safetensors:   0%|          |  550kB / 1.18GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpp6fscnr2/tokenizer.json       : 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/Harsha901/qwen2.5-1.5B-dpo-finetuned/commit/0cad9e82aa78399bf3a7cadf5a8dd8adc4409e80', commit_message='Upload tokenizer', commit_description='', oid='0cad9e82aa78399bf3a7cadf5a8dd8adc4409e80', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Harsha901/qwen2.5-1.5B-dpo-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Harsha901/qwen2.5-1.5B-dpo-finetuned'), pr_revision=None, pr_num=None)